# LoRA-Style 4-Bit Quantization Walkthrough

This notebook is based on the implementation in `src/lora-qlora/lora.py`. It explains the math behind the quantizer, then visualizes how quantization changes model parameters and how those parameter changes propagate into the final model output.

We will answer three questions:

1. How are the quantization codebook values constructed?
2. How much do individual parameters move after quantize + dequantize?
3. How much output error does that induce in the final forward pass?

## Quantization Math

The code in `lora.py` uses a small Gaussian-inspired codebook and a simple affine normalization. For a tensor $x$, let

$$
x_{\min} = in(x), \qquad x_{\max} = ax(x).
$$

It first maps values into an arbitrary range $[a, b] = [-1, 1]$:

$$
\tilde{x} = a + rac{x - x_{\min}}{x_{\max} - x_{\min}} (b - a).
$$

A codebook $\{c_1, \ldots, c_K\}$ is then built from Gaussian quantiles. This places more levels near the center than a uniform grid, which is useful when many weights are close to zero.

Each normalized value is quantized by nearest-neighbor lookup:

$$
q(\tilde{x}) = c_{k^*}, \qquad k^* = \arg\min_k |\tilde{x} - c_k|.
$$

Finally, dequantization maps the codebook value back to the original tensor scale:

$$
\hat{x} = rac{q(\tilde{x}) - a}{b - a}(x_{\max} - x_{\min}) + x_{\min}.
$$

To measure distortion we will use:

- Mean absolute error: $\mathrm{MAE}(x, \hat{x}) = \frac{1}{N} \sum_i |x_i - \hat{x}_i|$
- Root mean squared error: $\mathrm{RMSE}(x, \hat{x}) = \sqrt{\frac{1}{N} \sum_i (x_i - \hat{x}_i)^2}$
- Output drift: $\Delta y = f(x) - \hat{f}(x)$

In [ ]:
from pathlib import Path
import importlib.util

import matplotlib.pyplot as plt
import numpy as np
import torch

plt.style.use("seaborn-v0_8-whitegrid")
torch.manual_seed(7)
np.random.seed(7)

def load_lora_module():
    for root in [Path.cwd(), *Path.cwd().parents]:
        repo_candidate = root / "src" / "lora-qlora" / "lora.py"
        local_candidate = root / "lora.py"

        for candidate in (repo_candidate, local_candidate):
            if candidate.exists():
                spec = importlib.util.spec_from_file_location("lora_module", candidate)
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)
                return module, candidate

    raise FileNotFoundError("Could not locate lora.py from the current notebook location.")

lora_module, source_path = load_lora_module()
print(f"Loaded classes from: {source_path}")

In [ ]:
SimpleMap = lora_module.SimpleMap
Adaptor = lora_module.Adaptor
Quantization4bit = lora_module.Quantization4bit
QuantizedModel = lora_module.QuantizedModel

input_dim = 8
hidden_dim = input_dim * 4
layers = 4
rank = 4
batch_size = 256

x = torch.randn(batch_size, input_dim, dtype=torch.float16)
base_model = SimpleMap(input_dim, hidden_dim, layers).to(torch.float16)
adaptor = Adaptor(rank, input_dim).to(torch.float16)

baseline_output = base_model(x) + adaptor(x)

print("Base model parameter count:", sum(p.numel() for p in base_model.parameters()))
print("Adaptor parameter count:", sum(p.numel() for p in adaptor.parameters()))
print("Baseline output shape:", tuple(baseline_output.shape))

## Inspecting a Single Parameter Tensor

A good first step is to quantize one weight matrix and compare it with its dequantized version. The three plots below answer different questions:

- Histogram: does the value distribution become more clustered?
- Scatter plot: how closely does dequantization track the identity line?
- Error heatmap: where in the matrix is the distortion strongest?

In [ ]:
quantizer = Quantization4bit()
first_weight = next(base_model.parameters()).detach().clone()
quantized_first = quantizer.map_to_quantized_values(first_weight)
dequantized_first = quantizer.dequantize_values(quantized_first)
error_first = dequantized_first - first_weight

print("Codebook values:")
print(quantizer.codebook.float().cpu().numpy())
print()
print(f"Single tensor MAE:  {error_first.abs().mean().item():.6f}")
print(f"Single tensor RMSE: {torch.sqrt((error_first.float() ** 2).mean()).item():.6f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(first_weight.float().cpu().numpy().ravel(), bins=20, alpha=0.65, label="Original")
axes[0].hist(dequantized_first.float().cpu().numpy().ravel(), bins=20, alpha=0.65, label="Dequantized")
axes[0].set_title("Parameter distribution")
axes[0].set_xlabel("Weight value")
axes[0].legend()

x_points = first_weight.float().cpu().numpy().ravel()
y_points = dequantized_first.float().cpu().numpy().ravel()
min_val = min(x_points.min(), y_points.min())
max_val = max(x_points.max(), y_points.max())
axes[1].scatter(x_points, y_points, alpha=0.7, color="#0f766e")
axes[1].plot([min_val, max_val], [min_val, max_val], "k--", linewidth=1)
axes[1].set_title("Original vs dequantized")
axes[1].set_xlabel("Original weight")
axes[1].set_ylabel("Dequantized weight")

heatmap = axes[2].imshow(error_first.float().cpu().numpy(), cmap="coolwarm", aspect="auto")
axes[2].set_title("Elementwise error heatmap")
axes[2].set_xlabel("Input dimension")
axes[2].set_ylabel("Output dimension")
fig.colorbar(heatmap, ax=axes[2], fraction=0.046, pad=0.04, label="Error")

plt.tight_layout()
plt.show()

## Full-Model Parameter Distortion

Next we quantize every parameter tensor in the base model, dequantize them back to floating-point values, and compare each tensor against the original.

This separates *weight-space error* from *output-space error*. A layer can have noticeable parameter distortion but still produce modest output drift if the downstream network is not too sensitive in that region.

In [ ]:
quantized_model = QuantizedModel(base_model)
quantized_params = quantized_model.quantized_model_parameters()
dequantized_params = quantized_model.dequantized_model_parameters(quantized_params)
reconstructed_model = quantized_model.dequantize_model(quantized_params)

layer_metrics = []
for idx, (original_param, recovered_param) in enumerate(zip(base_model.parameters(), dequantized_params)):
    diff = recovered_param - original_param.detach()
    layer_metrics.append({
        "index": idx,
        "count": original_param.numel(),
        "mae": diff.abs().mean().item(),
        "rmse": torch.sqrt((diff.float() ** 2).mean()).item(),
        "max_abs": diff.abs().max().item(),
    })

for item in layer_metrics:
    print(
        f"Param {item['index']:02d} | count={item['count']:4d} | "
        f"MAE={item['mae']:.6f} | RMSE={item['rmse']:.6f} | MaxAbs={item['max_abs']:.6f}"
    )

indices = [item['index'] for item in layer_metrics]
mae_values = [item['mae'] for item in layer_metrics]
rmse_values = [item['rmse'] for item in layer_metrics]
max_values = [item['max_abs'] for item in layer_metrics]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].bar(indices, mae_values, color="#0f766e")
axes[0].set_title("Per-parameter MAE")
axes[0].set_xlabel("Parameter index")
axes[0].set_ylabel("MAE")

axes[1].bar(indices, rmse_values, color="#b45309")
axes[1].set_title("Per-parameter RMSE")
axes[1].set_xlabel("Parameter index")
axes[1].set_ylabel("RMSE")

axes[2].bar(indices, max_values, color="#1d4ed8")
axes[2].set_title("Per-parameter max absolute error")
axes[2].set_xlabel("Parameter index")
axes[2].set_ylabel("Max abs error")

plt.tight_layout()
plt.show()

## Final Output Error

Now we measure what matters most: the difference between the original final output and the output produced after replacing the base model with its quantized-then-dequantized version.

If the original final output is

$$
y = f_{\text{base}}(x) + f_{\text{adaptor}}(x),
$$

and the quantized version is

$$
\hat{y} = \hat{f}_{\text{base}}(x) + f_{\text{adaptor}}(x),
$$

then the output drift is

$$
\Delta y = \hat{y} - y.
$$

We visualize the distribution of $\Delta y$, the per-sample $L_2$ error, and the identity relationship between original and quantized outputs.

In [ ]:
with torch.no_grad():
    original_output = base_model(x) + adaptor(x)
    quantized_output = reconstructed_model(x) + adaptor(x)

output_diff = quantized_output - original_output
sample_l2 = torch.linalg.vector_norm(output_diff.float(), ord=2, dim=1)
feature_mae = output_diff.abs().float().mean(dim=0)

print(f"Final output MAE:      {output_diff.abs().mean().item():.6f}")
print(f"Final output RMSE:     {torch.sqrt((output_diff.float() ** 2).mean()).item():.6f}")
print(f"Final output max abs:  {output_diff.abs().max().item():.6f}")
print(f"Mean sample L2 error:  {sample_l2.mean().item():.6f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

axes[0].hist(output_diff.float().cpu().numpy().ravel(), bins=30, color="#1d4ed8", alpha=0.85)
axes[0].set_title("Distribution of final output error")
axes[0].set_xlabel("Quantized output - original output")

axes[1].plot(sample_l2.cpu().numpy(), color="#be123c")
axes[1].set_title("Per-sample L2 output error")
axes[1].set_xlabel("Sample index")
axes[1].set_ylabel(r"$||\Delta y||_2$")

original_flat = original_output.float().cpu().numpy().ravel()
quantized_flat = quantized_output.float().cpu().numpy().ravel()
min_output = min(original_flat.min(), quantized_flat.min())
max_output = max(original_flat.max(), quantized_flat.max())
axes[2].scatter(original_flat, quantized_flat, alpha=0.35, color="#047857")
axes[2].plot([min_output, max_output], [min_output, max_output], "k--", linewidth=1)
axes[2].set_title("Original vs quantized final output")
axes[2].set_xlabel("Original output")
axes[2].set_ylabel("Quantized output")

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(np.arange(feature_mae.numel()), feature_mae.cpu().numpy(), color="#7c3aed")
plt.title("Average absolute output error by feature")
plt.xlabel("Output feature index")
plt.ylabel("MAE")
plt.show()

## Reading the Results

Typical patterns to expect:

- The histogram becomes more clustered because many continuous weights collapse onto a small set of codebook values.
- The scatter plot bends into horizontal bands because multiple original values map to the same quantized level.
- Parameter MAE and RMSE are local weight-space distortions, while output error shows the end-to-end effect after the full forward pass.
- If output drift stays small even when parameter error is visible, the model is relatively robust to this quantization scheme for the chosen input distribution.

A useful next extension is to repeat the same experiment for different bit widths, different codebooks, or with larger batches and compare the error curves directly.